# K-ABENA — Noise Robustness : TF/Keras & HuggingFace (ch.16B)

**Mirror of notebook 13 — TensorFlow/Keras + HuggingFace Trainer integration.**

This notebook covers:
1. Label noise injection in TF/Keras pipelines
2. `KabenaCallback` noise-adaptive configuration
3. HuggingFace `Trainer` integration via custom loss
4. Side-by-side comparison: TF vs HuggingFace on noisy SST-2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yekoelite/kabena-ml/blob/main/tutorials/niveau1_notebook/14_noise_robustness_tf_hf.ipynb)
[![HuggingFace Space](https://img.shields.io/badge/🤗-Open%20in%20HF%20Spaces-blue)](https://huggingface.co/spaces/yekoelite/kabena-demo)


In [ ]:
# Install / Installation
# !pip install kabena-ml[tf] transformers datasets evaluate -q

import numpy as np
import pandas as pd
import tensorflow as tf
from kabena_ml.core.filter import calibrate_K
from kabena_ml.integrations.tf_utils import KabenaCallback, KabenaTFTrainer
from kabena_ml import KabenaConfig

print(f"TF version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU'))} GPU(s)")


## 1. Noise injection for TF pipelines / Injection de bruit pour TF

In [ ]:
import numpy as np

def inject_label_noise_np(labels: np.ndarray, noise_rate: float,
                           n_classes: int = 10, seed: int = 42) -> np.ndarray:
    """
    Inject symmetric label noise — NumPy/TF compatible version.
    / Injecter du bruit symétrique sur les labels — version NumPy/TF.

    Parameters / Paramètres
    ----------
    labels : np.ndarray   Clean integer labels / Labels entiers propres
    noise_rate : float    Fraction to corrupt [0.0, 1.0]
    n_classes : int       Number of classes / Nombre de classes

    Returns / Retourne
    -------
    np.ndarray : noisy labels / labels bruités
    """
    rng     = np.random.default_rng(seed)
    noisy   = labels.copy()
    n_noisy = int(len(labels) * noise_rate)
    indices = rng.choice(len(labels), size=n_noisy, replace=False)
    for idx in indices:
        orig      = int(noisy[idx])
        candidates = [c for c in range(n_classes) if c != orig]
        noisy[idx] = rng.choice(candidates)
    actual = (noisy != labels).mean()
    print(f"  Noise injected — requested: {noise_rate:.0%}, actual: {actual:.1%}")
    return noisy


# Test / Test rapide
y_clean = np.arange(10)
y_noisy = inject_label_noise_np(y_clean, noise_rate=0.30, n_classes=10)
print(f"  Original: {y_clean}")
print(f"  Noisy:    {y_noisy}")


## 2. TF/Keras — K-ABENA under label noise via KabenaCallback

In [ ]:
# --- TF/Keras noise robustness experiment ---
(X_tr, y_tr), (X_te, y_te) = tf.keras.datasets.cifar10.load_data()
MEAN = np.array([0.4914, 0.4822, 0.4465], dtype=np.float32)
STD  = np.array([0.2023, 0.1994, 0.2010], dtype=np.float32)
X_tr = ((X_tr.astype("float32") / 255) - MEAN) / STD
X_te = ((X_te.astype("float32") / 255) - MEAN) / STD
y_tr, y_te = y_tr.squeeze(), y_te.squeeze()


def augment(x, y):
    """Standard CIFAR-10 augmentation / Augmentation standard CIFAR-10."""
    x = tf.image.random_flip_left_right(x)
    x = tf.image.pad_to_bounding_box(x, 4, 4, 40, 40)
    x = tf.image.random_crop(x, [32, 32, 3])
    return x, y


def build_resnet_tf(n_classes=10):
    """ResNet-18 adapted for CIFAR in TF / ResNet-18 adapté CIFAR en TF."""
    def res_block(x, f, s=1):
        sc = x
        x = tf.keras.layers.Conv2D(f, 3, s, padding="same", use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.Conv2D(f, 3, 1, padding="same", use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        if s != 1 or sc.shape[-1] != f:
            sc = tf.keras.layers.Conv2D(f, 1, s, use_bias=False)(sc)
            sc = tf.keras.layers.BatchNormalization()(sc)
        return tf.keras.layers.ReLU()(x + sc)
    inp = tf.keras.Input(shape=(32, 32, 3))
    x = tf.keras.layers.Conv2D(64, 3, 1, padding="same", use_bias=False)(inp)
    x = tf.keras.layers.BatchNormalization()(x); x = tf.keras.layers.ReLU()(x)
    for f, nb, s in [(64,2,1),(128,2,2),(256,2,2),(512,2,2)]:
        x = res_block(x, f, s)
        for _ in range(nb-1): x = res_block(x, f)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    return tf.keras.Model(inp, tf.keras.layers.Dense(n_classes)(x))


EPOCHS_TF = 5  # Set to 50+ for book results / 50+ pour résultats du livre

def run_tf_noise_experiment(noise_rate: float, seed: int = 42):
    """
    Run TF/Keras experiment: standard vs K-ABENA under noise.
    / Expérience TF/Keras : standard vs K-ABENA sous bruit.
    """
    tf.random.set_seed(seed); np.random.seed(seed)

    # Inject noise into training labels
    # / Injecter le bruit dans les labels d'entraînement
    y_noisy = inject_label_noise_np(y_tr, noise_rate=noise_rate, n_classes=10, seed=seed)

    def make_ds(X, y, shuffle=False):
        ds = tf.data.Dataset.from_tensor_slices((X, y))
        if shuffle: ds = ds.shuffle(50000, seed=seed)
        ds = ds.batch(128)
        if shuffle: ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
        return ds.prefetch(tf.data.AUTOTUNE)

    train_ds = make_ds(X_tr, y_noisy, shuffle=True)
    test_ds  = make_ds(X_te, y_te)

    results = {}
    for variant in ["standard", "kabena"]:
        tf.random.set_seed(seed)
        model = build_resnet_tf()
        steps = len(X_tr) // 128
        lr_sc = tf.keras.optimizers.schedules.CosineDecay(0.1, EPOCHS_TF * steps)
        model.compile(
            optimizer=tf.keras.optimizers.SGD(lr_sc, momentum=0.9, weight_decay=1e-4),
            loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
            metrics=["accuracy"],
        )

        callbacks = []
        if variant == "kabena":
            # Noise-adaptive K calibration
            # / Calibrage K adaptatif au bruit
            sample_losses = []
            dummy = build_resnet_tf()
            for Xb, yb in train_ds.take(8):
                logits = dummy(Xb, training=False)
                l = tf.keras.losses.sparse_categorical_crossentropy(yb, logits, from_logits=True)
                sample_losses.extend(l.numpy())

            # Raise q to compensate for noise memorization
            # / Augmenter q pour compenser la mémorisation du bruit
            q     = min(10.0 + 2.0 * noise_rate * 100, 25.0)
            K_auto = float(np.percentile(sample_losses, q))
            N_use  = 0.3 if noise_rate < 0.10 else (0.2 if noise_rate < 0.30 else 0.0)

            print(f"    K-ABENA config: K={K_auto:.4f}, N={N_use}, q={q:.1f}%")
            callbacks.append(KabenaCallback(K=K_auto, N=N_use, verbose=False))

        hist = model.fit(
            train_ds, epochs=EPOCHS_TF, validation_data=test_ds,
            callbacks=callbacks, verbose=0,
        )
        val_acc = hist.history["val_accuracy"][-1] * 100
        results[variant] = val_acc
        print(f"  [{variant:10s}] noise={noise_rate:.0%} → acc={val_acc:.2f}%")

    return results


# Run across noise levels / Exécuter sur différents niveaux de bruit
noise_levels_tf = [0.0, 0.10, 0.20, 0.30]
tf_results = []

for noise in noise_levels_tf:
    print(f"\n{'='*50}\nNoise: {noise:.0%}")
    r = run_tf_noise_experiment(noise_rate=noise)
    tf_results.append({
        "Noise": f"{noise:.0%}",
        "TF Standard": f"{r['standard']:.2f}%",
        "TF K-ABENA":  f"{r['kabena']:.2f}%",
        "Δ":           f"{r['kabena']-r['standard']:+.2f} pts",
    })

print("\n=== TF/Keras Noise Robustness Results ===")
print(pd.DataFrame(tf_results).to_string(index=False))


## 3. HuggingFace Trainer — K-ABENA on noisy SST-2

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# HuggingFace integration — K-ABENA with custom training loop
# Intégration HuggingFace — K-ABENA avec boucle d'entraînement personnalisée
# ─────────────────────────────────────────────────────────────────────────────
import torch
import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import load_dataset
import evaluate as hf_evaluate

from kabena_ml.integrations.torch_utils import kabena_filter_torch

DEVICE_PT = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "distilbert-base-uncased"  # lighter than BERT-base for demo / plus léger pour démo

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
metric    = hf_evaluate.load("accuracy")


def tokenize(examples):
    """Tokenize SST-2 sentences / Tokenizer les phrases SST-2."""
    return tokenizer(examples["sentence"], truncation=True, max_length=128, padding="max_length")


# Load and tokenize SST-2 (subset for demo speed)
# / Charger et tokenizer SST-2 (sous-ensemble pour la démo)
sst2 = load_dataset("glue", "sst2")
sst2 = sst2.map(tokenize, batched=True)
sst2.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Demo subset / Sous-ensemble pour démo rapide
train_small = sst2["train"].select(range(2000))
val_small   = sst2["validation"].select(range(500))
print(f"SST-2 loaded — train: {len(train_small)}, val: {len(val_small)}")


In [ ]:
class KabenaTrainerHF(Trainer):
    """
    HuggingFace Trainer subclass with K-ABENA selective gradient computation.
    / Sous-classe HuggingFace Trainer avec calcul de gradient sélectif K-ABENA.

    Usage — identical to standard Trainer, with two additional parameters:
    / Usage — identique au Trainer standard, avec deux paramètres supplémentaires :
        kabena_K : float  — threshold / seuil (None = auto-calibrate at step 1)
        kabena_N : float  — retention proportion / proportion de rétention

    Integration cost vs standard Trainer: +3 lines in compute_loss.
    / Coût d'adoption vs Trainer standard : +3 lignes dans compute_loss.
    """
    def __init__(self, *args, kabena_K=None, kabena_N=0.3,
                 estimated_noise=0.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.kabena_K         = kabena_K      # None = auto-calibrate / None = calibrage auto
        self.kabena_N         = kabena_N      # retention proportion / proportion de rétention
        self.estimated_noise  = estimated_noise
        self._K_ema           = None
        self._alpha           = 0.05          # EMA decay / décroissance EMA
        self._step_count      = 0

    def compute_loss(self, model, inputs, return_outputs=False):
        """
        Override Trainer.compute_loss to apply K-ABENA filtering.
        / Surcharger Trainer.compute_loss pour appliquer le filtre K-ABENA.
        """
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits

        # --- Standard per-sample losses (reduction='none' is the key) ---
        # --- Pertes individuelles (reduction='none' est la clé) ---
        losses = F.cross_entropy(logits, labels, reduction="none")  # +1 vs standard

        # --- Auto-calibrate K at first step ---
        # --- Calibrage auto de K à la première étape ---
        if self._K_ema is None and self.kabena_K is None:
            q             = min(10.0 + 2.0 * self.estimated_noise * 100, 25.0)
            self._K_ema   = float(np.percentile(losses.detach().cpu().numpy(), q))
        elif self._K_ema is not None:
            q      = min(10.0 + 2.0 * self.estimated_noise * 100, 25.0)
            K_curr = float(np.percentile(losses.detach().cpu().numpy(), q))
            self._K_ema = self._alpha * K_curr + (1 - self._alpha) * self._K_ema

        K_use = self.kabena_K if self.kabena_K is not None else self._K_ema

        # --- K-ABENA filter — the 2 core lines ---
        # --- Filtre K-ABENA — les 2 lignes clés ---
        mask = kabena_filter_torch(losses, K=K_use, N=self.kabena_N)  # +1
        loss = losses[mask].mean() if mask.any() else losses.mean()   # +1

        self._step_count += 1
        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        """Standard eval — no K-ABENA filtering at inference / Eval standard sans filtre."""
        return super().prediction_step(model, inputs, prediction_loss_only, ignore_keys)


def compute_metrics(eval_pred):
    """Standard accuracy metric / Métrique accuracy standard."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)


In [ ]:
# Run HuggingFace experiments: standard vs K-ABENA under noise
# / Exécuter les expériences HF : standard vs K-ABENA sous bruit

training_args = TrainingArguments(
    output_dir="./hf_kabena_output",
    num_train_epochs=2,           # Set to 5+ for better results / 5+ pour meilleurs résultats
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    load_best_model_at_end=False,
    seed=42,
    report_to="none",             # Disable wandb/mlflow / Désactiver wandb/mlflow
)

hf_results = []

for noise_rate_hf in [0.0, 0.20]:   # Add 0.30, 0.40 for full study / Ajouter pour étude complète
    # Inject label noise into SST-2 training split
    # / Injecter le bruit dans le split train de SST-2
    labels_clean = np.array(train_small["label"])
    labels_noisy = inject_label_noise_np(labels_clean, noise_rate=noise_rate_hf,
                                         n_classes=2, seed=42)
    train_noisy_hf = train_small.map(
        lambda ex, idx: {"label": int(labels_noisy[idx])},
        with_indices=True
    )
    print(f"\n{'='*55}")
    print(f"HuggingFace SST-2 — noise: {noise_rate_hf:.0%}")
    print(f"{'='*55}")

    for variant in ["standard", "kabena"]:
        model_hf = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=2
        )

        if variant == "standard":
            # Standard HuggingFace Trainer — no modification
            # / Trainer HF standard — aucune modification
            trainer_hf = Trainer(
                model=model_hf,
                args=training_args,
                train_dataset=train_noisy_hf,
                eval_dataset=val_small,
                compute_metrics=compute_metrics,
            )
        else:
            # K-ABENA Trainer — subclass with noise-adaptive config
            # / Trainer K-ABENA — sous-classe avec config adaptative au bruit
            N_hf = 0.3 if noise_rate_hf < 0.10 else (0.2 if noise_rate_hf < 0.30 else 0.0)
            trainer_hf = KabenaTrainerHF(
                model=model_hf,
                args=training_args,
                train_dataset=train_noisy_hf,
                eval_dataset=val_small,
                compute_metrics=compute_metrics,
                kabena_K=None,                    # auto-calibrate / calibrage auto
                kabena_N=N_hf,                    # noise-adaptive N / N adaptatif
                estimated_noise=noise_rate_hf,    # used for q calibration / pour calibrage q
            )
            print(f"  K-ABENA config: N={N_hf}, noise={noise_rate_hf:.0%} (auto-K)")

        print(f"  Training [{variant}]...")
        train_result = trainer_hf.train()
        eval_result  = trainer_hf.evaluate()
        acc = eval_result["eval_accuracy"] * 100
        print(f"  [{variant:10s}] noise={noise_rate_hf:.0%} → acc={acc:.2f}%")
        hf_results.append({
            "Platform": "HuggingFace Trainer",
            "Variant": variant, "Noise": f"{noise_rate_hf:.0%}", "Acc": acc,
        })

print("\n=== HuggingFace Noise Robustness Results ===")
df_hf = pd.DataFrame(hf_results)
pivot = df_hf.pivot_table(index="Noise", columns="Variant", values="Acc")
pivot["Δ (pts)"] = pivot["kabena"] - pivot["standard"]
print(pivot.round(2).to_string())


## 4. Cross-platform comparison / Comparaison inter-plateforme

In [ ]:
# Summary: TF/Keras vs HuggingFace Trainer, both with K-ABENA
# / Synthèse : TF/Keras vs HuggingFace Trainer, tous deux avec K-ABENA

print("\n" + "="*70)
print("Cross-platform K-ABENA noise robustness summary")
print("Synthèse inter-plateforme robustesse K-ABENA sous bruit")
print("="*70)
print(f"{'Platform':<25} {'Noise':>8} {'Standard':>12} {'K-ABENA':>12} {'Δ':>10}")
print("-"*70)

# TF results
for r in tf_results:
    std_v = float(r["TF Standard"].rstrip("%"))
    ka_v  = float(r["TF K-ABENA"].rstrip("%"))
    print(f"{'TF/Keras':<25} {r['Noise']:>8} {std_v:>11.2f}% {ka_v:>11.2f}% {ka_v-std_v:>+9.2f}")

# HF results
for noise in ["0%", "20%"]:
    rows = [r for r in hf_results if r["Noise"] == noise]
    std_v = next(r["Acc"] for r in rows if r["Variant"] == "standard")
    ka_v  = next(r["Acc"] for r in rows if r["Variant"] == "kabena")
    print(f"{'HuggingFace Trainer':<25} {noise:>8} {std_v:>11.2f}% {ka_v:>11.2f}% {ka_v-std_v:>+9.2f}")

print("="*70)
print("\nAdoption cost comparison / Coût d'adoption comparé:")
print("  TF/Keras      : callbacks=[KabenaCallback(K, N)]  — +1 argument")
print("  HuggingFace   : Trainer → KabenaTrainerHF(...)    — +3 lines in compute_loss")
print("  PyTorch (nb13): kabena_filter_torch(losses, K, N) — +2 lines in loop")
print("\nKey insight / Observation clé:")
print("  K-ABENA benefit is consistent across all platforms.")
print("  The gain amplifies with noise regardless of framework.")
